# Phase 2: Load SEC Data into DuckDB with dlt

## Overview

Phase 2 takes the real financial data extracted from SEC XBRL API (Phase 1) and loads it into DuckDB using **dlt** (data load tool).

### What dlt Does:
- **Automatic schema management** - Creates tables with correct types
- **Duplicate detection** - Prevents re-loading same data
- **Data validation** - Catches errors before loading
- **Incremental updates** - Only loads new data
- **Built for DuckDB** - Optimized for local analytics

### Pipeline:
```
SEC API (Phase 1)
    ↓
  [Raw JSON]
    ↓
Transform (normalize + type hints)
    ↓
  [dlt Pipeline]
    ↓
DuckDB Tables (Phase 2) ← YOU ARE HERE
    ↓
SQL Queries + RAG (Phase 3)
```

---
## Setup: Import Dependencies

In [ ]:
import dlt
import duckdb
import logging
import pandas as pd
from typing import List, Dict, Any
from datetime import datetime
import sys

# Add learning directory to path so we can import our SEC API modules
sys.path.insert(0, '/Users/prajwalchambenandeeshappa/Github_Repos/Stocks_Earnings_Intelligence_Agent-Text2SQL/learning')

from sec_api_orchestrator import FinancialDataPipeline
from sec_api_endpoint1 import get_company_facts
from sec_api_endpoint2 import get_concept_over_time
from sec_api_endpoint3 import get_10q_filings_metadata

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Dependencies loaded")
print(f"   dlt version: {dlt.__version__}")
print(f"   DuckDB available")

---
## Part 1: Fetch Data from SEC API

Step 1: Use the SEC API modules to fetch real data

In [ ]:
# Companies to load
TICKERS = ["MSFT", "AAPL", "GOOGL"]

# Concepts to track
KEY_METRICS = [
    "Revenues",
    "NetIncomeLoss",
    "OperatingIncomeLoss",
    "Assets",
    "Liabilities",
    "StockholdersEquity",
]

print("🚀 Phase 2: Loading SEC Data into DuckDB")
print(f"\n📊 Companies: {', '.join(TICKERS)}")
print(f"📈 Metrics: {', '.join(KEY_METRICS[:3])}...")
print("\nFetching data from SEC XBRL API...\n")

In [ ]:
# Fetch data for each company
all_metrics_data = []  # For loading into DuckDB
all_filings_data = []  # For loading into DuckDB

for ticker in TICKERS:
    print(f"\n{'='*70}")
    print(f"Fetching data for {ticker}")
    print(f"{'='*70}")
    
    pipeline = FinancialDataPipeline(ticker)
    
    # Fetch metrics
    for metric in KEY_METRICS:
        logger.info(f"Fetching {metric}...")
        metric_data = pipeline.get_metric(metric, limit=10)
        
        # Add ticker to each record
        for record in metric_data:
            all_metrics_data.append(record)
    
    # Fetch filings metadata
    logger.info(f"Fetching filing metadata...")
    filings_data = pipeline.get_filings(limit=5)
    for filing in filings_data:
        all_filings_data.append(filing)

print(f"\n✅ Data collection complete!")
print(f"   Total metric records: {len(all_metrics_data)}")
print(f"   Total filing records: {len(all_filings_data)}")

---
## Part 2: Transform Data for dlt

Step 2: Convert raw SEC data into structured format with proper types

In [ ]:
# Show sample of raw data
print("📋 Sample metric record (raw from SEC API):\n")
print(all_metrics_data[0])

print("\n" + "="*70)
print("\n📋 Sample filing record (raw from SEC API):\n")
print(all_filings_data[0])

In [ ]:
def transform_metric_for_dlt(records: List[Dict]) -> List[Dict]:
    """
    Transform metric records for dlt loading.
    - Add load_timestamp for tracking when data was loaded
    - Convert string dates to proper date format
    - Ensure numeric fields are actual numbers
    """
    
    transformed = []
    load_time = datetime.now().isoformat()
    
    for record in records:
        transformed_record = {
            "ticker": record.get("ticker"),
            "concept": record.get("concept"),
            "period_end": record.get("period_end"),  # Will be converted to DATE by dlt
            "filing_date": record.get("filing_date"),  # Will be converted to DATE by dlt
            "form": record.get("form"),  # 10-Q, 10-K, etc.
            "value": int(record.get("value", 0)),  # Ensure it's an integer
            "load_timestamp": load_time,  # When we loaded it
        }
        transformed.append(transformed_record)
    
    return transformed


def transform_filings_for_dlt(records: List[Dict]) -> List[Dict]:
    """
    Transform filing records for dlt loading.
    """
    
    transformed = []
    load_time = datetime.now().isoformat()
    
    for record in records:
        transformed_record = {
            "ticker": record.get("ticker"),
            "form": record.get("form"),  # 10-Q, 10-K
            "filing_date": record.get("filing_date"),  # When filed
            "period_end": record.get("period_end"),  # End of reporting period
            "accession_number": record.get("accession_number"),  # SEC's ID
            "filing_url": record.get("filing_url"),  # URL to filing
            "load_timestamp": load_time,
        }
        transformed.append(transformed_record)
    
    return transformed


# Transform the data
metrics_transformed = transform_metric_for_dlt(all_metrics_data)
filings_transformed = transform_filings_for_dlt(all_filings_data)

print("✅ Data transformation complete")
print(f"\n📋 Transformed metric record:\n")
print(metrics_transformed[0])

print(f"\n📋 Transformed filing record:\n")
print(filings_transformed[0])

---
## Part 3: Create dlt Pipeline and Load into DuckDB

Step 3: Set up dlt with DuckDB and load the transformed data

In [ ]:
# Define the dlt pipeline
# dlt will automatically:
# - Create tables if they don't exist
# - Infer schemas from the data
# - Handle duplicates
# - Track load history

# Create a pipeline that loads into DuckDB
# This is stored locally in a .duckdb file
pipeline = dlt.pipeline(
    pipeline_name="financial_data_pipeline",
    destination="duckdb",
    dataset_name="sec_filings"
)

print("✅ dlt pipeline created")
print(f"   Pipeline name: financial_data_pipeline")
print(f"   Destination: DuckDB")
print(f"   Dataset: sec_filings")
print(f"   Database file: ~/.dlt/pipelines/financial_data_pipeline/sec_filings.duckdb")

In [ ]:
# Load metrics data into DuckDB
# dlt will create a "financial_metrics" table

print("\n📊 Loading financial metrics into DuckDB...")

load_info = pipeline.run(
    metrics_transformed,
    table_name="financial_metrics",
    write_disposition="append",  # Append to existing table (no duplicates)
)

print(f"\n✅ Metrics loaded!")
print(f"   Records loaded: {len(metrics_transformed)}")
print(f"   Table: financial_metrics")

In [ ]:
# Load filings data into DuckDB
# dlt will create a "sec_filings" table

print("\n📄 Loading SEC filings metadata into DuckDB...")

load_info = pipeline.run(
    filings_transformed,
    table_name="sec_filings_metadata",
    write_disposition="append",
)

print(f"\n✅ Filings loaded!")
print(f"   Records loaded: {len(filings_transformed)}")
print(f"   Table: sec_filings_metadata")

---
## Part 4: Verify Data in DuckDB

Step 4: Connect to DuckDB and verify the loaded data

In [ ]:
# Connect to the DuckDB database that dlt created
db_path = pipeline.default_schema.connection.db_path
print(f"📂 Database location: {db_path}")

# Open a connection
conn = duckdb.connect(db_path)

# Show all tables
tables = conn.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'sec_filings'").fetchall()

print(f"\n📊 Tables in database:")
for table in tables:
    print(f"   - {table[0]}")

In [ ]:
# Query the financial_metrics table
print("\n" + "="*70)
print("FINANCIAL METRICS TABLE")
print("="*70)

metrics_df = conn.execute("""
    SELECT * FROM sec_filings.financial_metrics
    LIMIT 10
""").df()

print(f"\nFirst 10 rows:")
print(metrics_df.to_string())

print(f"\n📊 Table info:")
schema = conn.execute("DESCRIBE sec_filings.financial_metrics").df()
print(schema)

In [ ]:
# Get summary statistics
print("\n" + "="*70)
print("METRICS SUMMARY")
print("="*70)

summary = conn.execute("""
    SELECT 
        concept,
        COUNT(*) as count,
        COUNT(DISTINCT ticker) as unique_companies,
        COUNT(DISTINCT form) as form_types,
        MIN(value) as min_value,
        MAX(value) as max_value,
        AVG(value) as avg_value
    FROM sec_filings.financial_metrics
    GROUP BY concept
    ORDER BY count DESC
""").df()

print(summary.to_string())

In [ ]:
# Query by company
print("\n" + "="*70)
print("DATA BY COMPANY")
print("="*70)

by_company = conn.execute("""
    SELECT 
        ticker,
        COUNT(*) as records,
        COUNT(DISTINCT concept) as concepts,
        COUNT(DISTINCT form) as form_types,
        MIN(period_end) as earliest_period,
        MAX(period_end) as latest_period
    FROM sec_filings.financial_metrics
    GROUP BY ticker
    ORDER BY ticker
""").df()

print(by_company.to_string())

In [ ]:
# Query the filings metadata table
print("\n" + "="*70)
print("SEC FILINGS METADATA TABLE")
print("="*70)

filings_df = conn.execute("""
    SELECT * FROM sec_filings.sec_filings_metadata
    LIMIT 10
""").df()

print(f"\nFirst 10 rows:")
print(filings_df.to_string())

print(f"\n📊 Table info:")
schema = conn.execute("DESCRIBE sec_filings.sec_filings_metadata").df()
print(schema)

---
## Part 5: Create Optimized Indexes

Step 5: Add indexes for fast queries

In [ ]:
print("\n" + "="*70)
print("CREATING INDEXES FOR FAST QUERIES")
print("="*70)

# Index on ticker for company queries
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_metrics_ticker 
    ON sec_filings.financial_metrics(ticker)
""")
print("✅ Created index: ticker")

# Index on concept for metric queries
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_metrics_concept 
    ON sec_filings.financial_metrics(concept)
""")
print("✅ Created index: concept")

# Index on period_end for time series queries
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_metrics_period 
    ON sec_filings.financial_metrics(period_end)
""")
print("✅ Created index: period_end")

# Index on form type
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_metrics_form 
    ON sec_filings.financial_metrics(form)
""")
print("✅ Created index: form")

# Composite index for common queries
conn.execute("""
    CREATE INDEX IF NOT EXISTS idx_metrics_ticker_concept 
    ON sec_filings.financial_metrics(ticker, concept)
""")
print("✅ Created index: (ticker, concept)")

print("\n✅ All indexes created successfully")

---
## Part 6: Example Queries

Step 6: Show how to query the loaded data

In [ ]:
# Query 1: Apple's Revenue Trend
print("\n" + "="*70)
print("QUERY 1: Apple's Revenue Trend")
print("="*70)

apple_revenue = conn.execute("""
    SELECT 
        period_end,
        form,
        filing_date,
        value as revenue,
        value / 1000000000 as revenue_billions
    FROM sec_filings.financial_metrics
    WHERE ticker = 'AAPL' 
      AND concept = 'Revenues'
    ORDER BY period_end DESC
    LIMIT 5
""").df()

print(apple_revenue.to_string())

In [ ]:
# Query 2: Compare Net Income Across Companies
print("\n" + "="*70)
print("QUERY 2: Net Income Comparison (Most Recent)")
print("="*70)

net_income_comp = conn.execute("""
    SELECT 
        ticker,
        MAX(period_end) as latest_period,
        MAX(CASE WHEN form = '10-Q' THEN value ELSE NULL END) as latest_10q_net_income,
        MAX(CASE WHEN form = '10-K' THEN value ELSE NULL END) as latest_10k_net_income,
        MAX(value) / 1000000000 as max_net_income_billions
    FROM sec_filings.financial_metrics
    WHERE concept = 'NetIncomeLoss'
    GROUP BY ticker
    ORDER BY ticker
""").df()

print(net_income_comp.to_string())

In [ ]:
# Query 3: Total Assets Over Time
print("\n" + "="*70)
print("QUERY 3: Total Assets by Company (10-K Annual Only)")
print("="*70)

total_assets = conn.execute("""
    SELECT 
        ticker,
        period_end,
        form,
        value / 1000000000 as assets_billions
    FROM sec_filings.financial_metrics
    WHERE concept = 'Assets' 
      AND form = '10-K'
    ORDER BY ticker, period_end DESC
    LIMIT 3
""").df()

print(total_assets.to_string())

In [ ]:
# Query 4: Filing Timeline
print("\n" + "="*70)
print("QUERY 4: Recent Filings Timeline")
print("="*70)

filings_timeline = conn.execute("""
    SELECT 
        ticker,
        form,
        filing_date,
        period_end,
        accession_number
    FROM sec_filings.sec_filings_metadata
    ORDER BY filing_date DESC
    LIMIT 10
""").df()

print(filings_timeline.to_string())

In [ ]:
# Query 5: Financial Health Snapshot (Current Quarter)
print("\n" + "="*70)
print("QUERY 5: Financial Health Snapshot (Most Recent)")
print("="*70)

health_snapshot = conn.execute("""
    SELECT 
        ticker,
        MAX(period_end) as period,
        MAX(CASE WHEN concept = 'Revenues' THEN value END) / 1000000000 as revenue_billions,
        MAX(CASE WHEN concept = 'NetIncomeLoss' THEN value END) / 1000000000 as net_income_billions,
        MAX(CASE WHEN concept = 'Assets' THEN value END) / 1000000000 as assets_billions,
        MAX(CASE WHEN concept = 'StockholdersEquity' THEN value END) / 1000000000 as equity_billions,
        ROUND(
            MAX(CASE WHEN concept = 'NetIncomeLoss' THEN value END) * 100.0 
            / MAX(CASE WHEN concept = 'Revenues' THEN value END)
        , 1) as profit_margin_pct
    FROM sec_filings.financial_metrics
    WHERE concept IN ('Revenues', 'NetIncomeLoss', 'Assets', 'StockholdersEquity')
    GROUP BY ticker
    ORDER BY ticker
""").df()

print(health_snapshot.to_string())

---
## Part 7: Export for RAG Pipeline

Step 7: Prepare data for Phase 3 (RAG)

In [ ]:
# Export recent data as JSON for RAG pipeline
print("\n" + "="*70)
print("EXPORTING DATA FOR RAG PIPELINE")
print("="*70)

# Get recent financial data
rag_data = conn.execute("""
    SELECT 
        ticker,
        concept,
        period_end,
        form,
        value,
        filing_date
    FROM sec_filings.financial_metrics
    WHERE period_end >= '2024-01-01'  -- Recent data only
    ORDER BY ticker, period_end DESC
""").df()

# Convert to records for JSON export
rag_records = rag_data.to_dict('records')

import json

# Save to file
output_file = '/Users/prajwalchambenandeeshappa/Github_Repos/Stocks_Earnings_Intelligence_Agent-Text2SQL/learning/financial_data_for_rag.json'
with open(output_file, 'w') as f:
    json.dump(rag_records, f, indent=2, default=str)

print(f"\n✅ Exported {len(rag_records)} records to:")
print(f"   {output_file}")
print(f"\n📊 Data summary:")
print(f"   Date range: {rag_data['period_end'].min()} to {rag_data['period_end'].max()}")
print(f"   Companies: {rag_data['ticker'].nunique()}")
print(f"   Metrics: {rag_data['concept'].nunique()}")

---
## Part 8: Summary

✅ **Phase 2 Complete!**

In [ ]:
print("\n" + "="*70)
print("PHASE 2 SUMMARY")
print("="*70)

# Final statistics
final_stats = conn.execute("""
    SELECT 
        'Metrics Table' as table_name,
        COUNT(*) as record_count,
        COUNT(DISTINCT ticker) as unique_tickers,
        COUNT(DISTINCT concept) as unique_concepts
    FROM sec_filings.financial_metrics
    UNION ALL
    SELECT 
        'Filings Table' as table_name,
        COUNT(*) as record_count,
        COUNT(DISTINCT ticker) as unique_tickers,
        COUNT(DISTINCT form) as unique_forms
    FROM sec_filings.sec_filings_metadata
""").df()

print("\n📊 Database Statistics:")
print(final_stats.to_string())

print("\n" + "="*70)
print("✅ WHAT WE ACCOMPLISHED:")
print("="*70)
print("""
1. ✅ Fetched real SEC data from 3 endpoints
2. ✅ Transformed data for dlt (added types, timestamps)
3. ✅ Loaded into DuckDB using dlt pipeline
4. ✅ Created optimized indexes
5. ✅ Ran example queries
6. ✅ Exported data for RAG pipeline

📊 Tables Created:
   • financial_metrics (time series financial data)
   • sec_filings_metadata (filing dates and URLs)

🔍 Indexes Created:
   • idx_metrics_ticker
   • idx_metrics_concept
   • idx_metrics_period
   • idx_metrics_form
   • idx_metrics_ticker_concept

📈 Ready for Phase 3:
   • Financial data is now in DuckDB
   • Can run SQL queries
   • Ready for RAG pipeline integration
   • Data exported to JSON for Claude integration
""")

print("="*70)
print("\n🚀 Next: Phase 3 - RAG Pipeline Integration")
print("   See: PHASE_3_RAG_PIPELINE.ipynb")
print("="*70)

---
## Cleanup

Close database connection

In [ ]:
# Close connection
conn.close()
print("✅ Database connection closed")
print("\n📂 Database files:")
print(f"   Location: ~/.dlt/pipelines/financial_data_pipeline/")
print(f"   File: sec_filings.duckdb")